In [0]:

%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
##Test

In [0]:

jdbc_url = (
    "jdbc:sqlserver://complaints-sql-server.database.windows.net:1433;"
    "databaseName=customer-complaints-db;"
    "encrypt=true;trustServerCertificate=false;loginTimeout=30"
)
jdbc_props = {
    "user":     dbutils.secrets.get(scope="complaints_jdbc", key="sql_user"),
    "password": dbutils.secrets.get(scope="complaints_jdbc", key="sql_password"),
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

In [0]:
target = "bronze.customer_complaints"

from pyspark.sql.functions import current_timestamp, lit, max as spark_max

if spark.catalog.tableExists(target):
    last_date = spark.table(target).agg(spark_max("Date_received")).collect()[0][0]
    print(f"Incremental load — rows on/after {last_date}")
else:
    last_date = "1900-01-01"
    print("First run — full load")

In [0]:

query = f"(SELECT * FROM dbo.customer_complaints WHERE Date_received >= '{last_date}') AS batch"

batch = (
    spark.read.jdbc(url=jdbc_url, table=query, properties=jdbc_props)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("azure_sql"))
)
batch.createOrReplaceTempView("batch")
print(f"Rows pulled (includes boundary re-check): {batch.count()}")

In [0]:

if spark.catalog.tableExists(target):
    inserted = spark.sql(f"""
        MERGE INTO {target} AS t
        USING batch AS s
        ON t.Complaint_ID = s.Complaint_ID
        WHEN NOT MATCHED THEN INSERT *
    """).collect()[0]["num_inserted_rows"]
else:
    batch.write.format("delta").mode("overwrite").saveAsTable(target)
    inserted = batch.count()

print(f"Inserted {inserted} new rows")

In [0]:
%sql
select * from bronze.customer_complaints limit 10